# AI-Based Cyber Threat Detection

This notebook is the main academic workflow for the AI in Cyber Security Lab. It will use the real dataset placed in the `data/` directory.

Task 5, ethical reflection and AI challenges, is intentionally excluded. No ethical simulation functionality is included.

## Planned Workflow

1. Import libraries
2. Load and inspect the actual dataset
3. Clean and explore the data
4. Preprocess features after inspecting their types
5. Train and evaluate the basic machine-learning model
6. Train and evaluate the simple MLP prototype
7. Compare actual model results

Implementation will be added task by task. No results are claimed in this outline.

## Dataset Source

The dataset source, file format, target column, and any limitations will be recorded here after the dataset is inspected.

## Development Boundary

This notebook currently contains only the project outline. Dataset loading, EDA, model training, evaluation, and plot generation will be implemented only when the corresponding task is started.

## 1. Import Libraries and Project Modules

The project modules provide reusable loading, EDA, preprocessing, modeling, and comparison functions.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATASET_PATH, METRICS_DIR, MODELS_DIR, PLOTS_DIR, RANDOM_STATE
from src.data_loader import inspect_nsl_kdd, load_nsl_kdd
from src.preprocessing import add_binary_target, split_dataset
from src.eda import (
    save_attack_type_distribution,
    save_class_distribution,
    save_correlation_heatmap,
    save_feature_distributions,
)

sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")

## 2. Load the Actual Dataset

`KDDTrain+.txt` is a headerless, comma-separated NSL-KDD file. The loader applies the documented 43-column schema.

In [ ]:
data = load_nsl_kdd(DATASET_PATH)
inspection = inspect_nsl_kdd(data)

print(f"Rows: {data.shape[0]}")
print(f"Columns: {data.shape[1]}")
print(f"Delimiter: comma (,)")
print(data.head())

## 3. Dataset Inspection

The following checks describe the actual dataset before any preprocessing or modeling.

In [ ]:
print("Shape:", inspection["shape"])
print("Columns:")
print(inspection["columns"])
print("Data types:")
print(data.dtypes)
print("Missing values:")
print(data.isna().sum())
print("Duplicate rows:", inspection["duplicate_rows"])
print("Basic statistics:")
display(data.describe(include="all").transpose())
print("Unique labels:", inspection["target_unique_values"])
print("Class distribution:")
display(data["label"].value_counts().to_frame("count"))
print("Numerical features:", inspection["numeric_features"])
print("Categorical features:", inspection["categorical_features"])

## 4. Exploratory Data Analysis

For EDA, `normal` remains the normal class and every other original NSL-KDD label is grouped as `Attack`. The original label column is preserved for attack-type analysis.

In [ ]:
eda_data = add_binary_target(data)

save_class_distribution(eda_data, "behavior", PLOTS_DIR / "normal_vs_attack.png")
save_attack_type_distribution(eda_data, PLOTS_DIR / "attack_type_frequency.png")
save_feature_distributions(
    eda_data,
    ["src_bytes", "dst_bytes", "count", "dst_host_srv_count"],
    PLOTS_DIR / "important_feature_distributions.png",
)
save_correlation_heatmap(
    eda_data,
    ["duration", "src_bytes", "dst_bytes", "count", "srv_count", "same_srv_rate", "dst_host_srv_count"],
    PLOTS_DIR / "numeric_correlation_heatmap.png",
)

print(eda_data["behavior"].value_counts())

### EDA Observations

- The Normal/Attack chart shows the balance of routine and suspicious records in the benchmark.
- The attack-type chart shows which named event labels occur most often; these are dataset frequencies, not live incident rates.
- The feature histograms show how traffic volume and connection-count variables vary across the records.
- The correlation heatmap highlights relationships among selected numerical indicators.

NSL-KDD is a benchmark dataset and does not perfectly represent modern real-world network traffic.

## 5. Preprocessing and Train/Test Split

The target is `behavior`. Original `label` and `difficulty_level` are excluded from model features to avoid target leakage. Numeric missing values are median-imputed and categorical values are one-hot encoded.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(
    eda_data,
    target_column="behavior",
    random_state=RANDOM_STATE,
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training classes:")
print(y_train.value_counts())

## 6. Decision Tree Classifier

This is the basic supervised machine-learning model. Metrics below will be calculated from the actual test predictions when the cell is run.

In [ ]:
from src.compare_models import evaluate_model, save_comparison
from src.ml_model import train_decision_tree

tree_model, tree_time = train_decision_tree(
    X_train,
    y_train,
    random_state=RANDOM_STATE,
    model_path=MODELS_DIR / "decision_tree.joblib",
)
tree_metrics = evaluate_model(
    tree_model,
    X_test,
    y_test,
    "Decision Tree",
    tree_time,
    METRICS_DIR,
    PLOTS_DIR,
)
print(tree_metrics)

## 7. MLP Neural Network

The MLP uses the same processed features and the same train/test split. It is a small introductory neural-network prototype, not a complex deep-learning system.

In [ ]:
from src.dl_model import train_mlp

mlp_model, mlp_time = train_mlp(
    X_train,
    y_train,
    random_state=RANDOM_STATE,
    model_path=MODELS_DIR / "mlp_classifier.joblib",
)
mlp_metrics = evaluate_model(
    mlp_model,
    X_test,
    y_test,
    "MLP Neural Network",
    mlp_time,
    METRICS_DIR,
    PLOTS_DIR,
)
print(mlp_metrics)

## 8. Model Comparison

All values in the comparison table come from the two actual test runs. No metric values are hard-coded.

In [ ]:
comparison = save_comparison(
    [tree_metrics, mlp_metrics],
    METRICS_DIR,
    PLOTS_DIR,
)
display(comparison)

## 9. Final Observations

- The dataset contains normal events and multiple named attack categories.
- The binary target simplifies the lab question to Normal versus Attack.
- The Decision Tree and MLP use the same split for a fair comparison.
- Accuracy, precision, recall, F1-score, and training time must be interpreted together.
- NSL-KDD is a benchmark dataset and does not perfectly represent current real-world traffic.

The project intentionally stops at Tasks 1–4. Task 5 is not implemented.